In [6]:
train_data_path = '../../../data/predictor/cof/rth_1.0/features/features_rth_1.0_ecfp_2_1024_False_descriptor__custom_pattern_False.csv'
train_data_feature_types = '../../../data/predictor/cof/rth_1.0/feature_types/feature_types_rth_1.0_ecfp_2_1024_False_descriptor__custom_pattern_False.yaml'

In [7]:
import pandas as pd
import yaml

df_train = pd.read_csv(train_data_path)
with open(train_data_feature_types, 'r') as f:
    train_data_feature_types = yaml.safe_load(f)

In [11]:
import os
import copy
from modules.predictor.training_and_evaluation.sklearn_pipeline import SklearnTrainingPipeline
from modules.predictor.data.utils import custom_data_kfold

X = df_train.drop(columns=["capacity_max", 'smiles'])
y = df_train[["capacity_max"]]
folds = custom_data_kfold(X, y, num_splits=10, num_bins=10, random_state=42)
model = 'xgboost'
d_name = 'cof_rth_1.0_ecfp_2_1024_False_descriptor_custom_pattern_False'
results_dir = 'results'

save_dir = os.path.join(results_dir, model, d_name)
print(d_name, model, save_dir)
os.makedirs(save_dir, exist_ok=True)

cof_rth_1.0_ecfp_2_1024_False_descriptor_custom_pattern_False xgboost results/xgboost/cof_rth_1.0_ecfp_2_1024_False_descriptor_custom_pattern_False


In [12]:
pipeline = SklearnTrainingPipeline(
    X=copy.deepcopy(X),
    y=copy.deepcopy(y),
    feature_types=train_data_feature_types,
    folds=copy.deepcopy(folds),
    metrics=['rmse', 'mae', 'r2', 'pairwise_accuracy_score'],
    save_dir=save_dir,
    data_name=d_name,
    hyperparam_opt=True,
    num_bins=10,
    verbose=True,
)

In [13]:
model = pipeline.train_and_save_model(model)

Training final model XGBoost Regressor on the entire dataset
Best score: 172.21936396734284
 Best params: {'n_estimators': 45, 'learning_rate': 0.05, 'max_depth': 11, 'min_child_weight': 3, 'gamma': 0.1, 'colsample_bytree': 0.3}
Model saved to results/xgboost/cof_rth_1.0_ecfp_2_1024_False_descriptor_custom_pattern_False/models/model_final.joblib


In [14]:
test_data_path = '../../../data/generated/ceteefy.csv'
cof_data_path = '../../../data/predictor/cof/data_substrate_cofs.csv'

test_data = pd.read_csv(test_data_path)
cof_data = pd.read_csv(cof_data_path)

test_data.head()

,canon_smiles,smarts_filter,conjugation_filter,flatness,normalized_csm,similarity,steric_hindrance,selfies
0,N#Cc1ccc(C#N)cc1,1,1,0.005030,0.000860,0.390080,1,[N][#C][C][=C][C][=C][Branch1][Ring1][C][#N][C...
1,N#Cc1ccc(-n2c(-c3c(F)c(F)cc(F)c3F)cc3c2cc(-c2c...,1,1,1.300015,0.067548,0.385453,1,[N][#C][C][=C][C][=C][Branch2][=Branch1][Ring1...
2,N#Cc1ccc(-n2c(=O)c3cc4c(=O)n(-c5ccc(C#N)cc5)c(...,1,1,0.706897,0.000203,0.374693,1,[N][#C][C][=C][C][=C][Branch2][Ring2][=N][N][C...
3,N#Cc1ccc(-n2[nH]c(=O)c3cc4c(=O)[nH]n(-c5ccc(C#...,1,1,0.697726,0.000662,0.319701,1,[N][#C][C][=C][C][=C][Branch2][Ring2][#C][N][N...
4,N#Cc1ccc(-n2c(-c3cc4ccccc4s3)cc3c2cc(-c2cc4ccc...,1,1,1.030009,0.000083,0.344513,1,[N][#C][C][=C][C][=C][Branch2][Branch1][#Branc...


In [15]:
test_data = test_data[['canon_smiles']]
test_data = test_data.rename(columns={'canon_smiles': 'smiles'})
test_data['capacity_max'] = 0
test_data.head()

,smiles,capacity_max
0,N#Cc1ccc(C#N)cc1,0
1,N#Cc1ccc(-n2c(-c3c(F)c(F)cc(F)c3F)cc3c2cc(-c2c...,0
2,N#Cc1ccc(-n2c(=O)c3cc4c(=O)n(-c5ccc(C#N)cc5)c(...,0
3,N#Cc1ccc(-n2[nH]c(=O)c3cc4c(=O)[nH]n(-c5ccc(C#...,0
4,N#Cc1ccc(-n2c(-c3cc4ccccc4s3)cc3c2cc(-c2cc4ccc...,0


In [16]:
concat_data = pd.concat([cof_data, test_data], axis=0, ignore_index=True)
concat_data

,smiles,capacity_max
0,N#Cc1ccc(C#N)cc1,628.0
1,CC(C)c1ccc(-n2c(-c3ccc(C#N)cc3)cc3c2cc(-c2ccc(...,255.0
2,Cc1ccc2nc(-n3c(-c4ccc(C#N)cc4)cc4c3cc(-c3ccc(C...,269.0
3,Clc1nc(Cl)nc(-c2ccc(-c3cc(-c4ccc(-c5nc(Cl)nc(C...,228.0
4,O=C1c2ccc(Nc3nc(Cl)nc(Cl)n3)cc2C(=O)c2ccc(Nc3n...,468.0
...,...,...
2748,N#Cc1ccc(-n2c(=O)c3cc4c(=O)n(-c5ccc(C#N)c(-c6c...,0.0
2749,N#CC(C#N)=c1cc2c(=C(C#N)C#N)cc1-2,0.0
2750,N#CC(C#N)=c1ccc(=C(C#N)C#N)c2c1C2,0.0
2751,CC1c2c1c(=C(C#N)C#N)ccc2=C(C#N)C#N,0.0


In [21]:
from modules.predictor.features.custom_descriptor import *
from modules.predictor.features.custom_patterns import *
from modules.predictor.features.feature_factory import *
from modules.predictor.features.fingerprints import *

data_types = [ "ecfp", "descriptor", "custom_pattern" ]
data_params = {
        "ecfp": {
            "radius": 2,
            "size": 1024,
            "count": False
        },
        "descriptor": {
        },
        "custom_pattern": {
            "count": False
        }
    }
kwargs = {'kwargs': data_params}
concat_data_features = generate_features(concat_data, {}, 'smiles', 'capacity_max', data_types, 1.0, **kwargs)
concat_data_features

2753it [00:02, 1198.77it/s]


(      ecfp_1  ecfp_2  ecfp_3  ecfp_4  ecfp_5  ecfp_6  ecfp_7  ecfp_8  ecfp_9  \
 0        0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
 1        1.0     0.0     0.0     1.0     0.0     0.0     0.0     0.0     0.0   
 2        0.0     0.0     0.0     1.0     0.0     0.0     0.0     0.0     0.0   
 3        0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
 4        1.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
 ...      ...     ...     ...     ...     ...     ...     ...     ...     ...   
 2748     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
 2749     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
 2750     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
 2751     0.0     0.0     0.0     0.0     0.0     0.0     1.0     0.0     0.0   
 2752     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
 
       ecfp_10  ...       

In [22]:
concat_data_features, f_types = concat_data_features
columns_to_use = df_train.drop(columns=['capacity_max', 'smiles']).columns.tolist()
concat_data_features = concat_data_features[columns_to_use + ['smiles', 'capacity_max']]
concat_data_features

,ecfp_1,ecfp_4,ecfp_7,ecfp_8,ecfp_14,ecfp_15,ecfp_17,ecfp_18,ecfp_20,ecfp_21,...,c%,custom_pattern_0,custom_pattern_1,custom_pattern_2,custom_pattern_3,custom_pattern_4,custom_pattern_5,custom_pattern_6,smiles,capacity_max
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.800000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,N#Cc1ccc(C#N)cc1,628.0
1,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.904762,0.0,0.0,0.0,0.0,1.0,0.0,0.0,CC(C)c1ccc(-n2c(-c3ccc(C#N)cc3)cc3c2cc(-c2ccc(...,255.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.818182,0.0,0.0,0.0,0.0,1.0,0.0,0.0,Cc1ccc2nc(-n3c(-c4ccc(C#N)cc4)cc4c3cc(-c3ccc(C...,269.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.687500,0.0,1.0,0.0,1.0,0.0,0.0,0.0,Clc1nc(Cl)nc(-c2ccc(-c3cc(-c4ccc(-c5nc(Cl)nc(C...,228.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.588235,0.0,0.0,0.0,1.0,0.0,0.0,0.0,O=C1c2ccc(Nc3nc(Cl)nc(Cl)n3)cc2C(=O)c2ccc(Nc3n...,468.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2748,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.769231,0.0,1.0,0.0,0.0,1.0,0.0,0.0,N#Cc1ccc(-n2c(=O)c3cc4c(=O)n(-c5ccc(C#N)c(-c6c...,0.0
2749,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.750000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,N#CC(C#N)=c1cc2c(=C(C#N)C#N)cc1-2,0.0
2750,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.764706,0.0,0.0,0.0,0.0,0.0,0.0,0.0,N#CC(C#N)=c1ccc(=C(C#N)C#N)c2c1C2,0.0
2751,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.777778,0.0,0.0,0.0,0.0,0.0,0.0,0.0,CC1c2c1c(=C(C#N)C#N)ccc2=C(C#N)C#N,0.0


In [24]:
test_data_features = concat_data_features.iloc[cof_data.shape[0]: , :].reset_index(drop=True)
test_data_features

,ecfp_1,ecfp_4,ecfp_7,ecfp_8,ecfp_14,ecfp_15,ecfp_17,ecfp_18,ecfp_20,ecfp_21,...,c%,custom_pattern_0,custom_pattern_1,custom_pattern_2,custom_pattern_3,custom_pattern_4,custom_pattern_5,custom_pattern_6,smiles,capacity_max
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.800000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,N#Cc1ccc(C#N)cc1,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.727273,0.0,0.0,0.0,0.0,1.0,0.0,0.0,N#Cc1ccc(-n2c(-c3c(F)c(F)cc(F)c3F)cc3c2cc(-c2c...,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.750000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,N#Cc1ccc(-n2c(=O)c3cc4c(=O)n(-c5ccc(C#N)cc5)c(...,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.705882,0.0,0.0,0.0,0.0,1.0,0.0,0.0,N#Cc1ccc(-n2[nH]c(=O)c3cc4c(=O)[nH]n(-c5ccc(C#...,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.857143,0.0,0.0,0.0,0.0,1.0,0.0,0.0,N#Cc1ccc(-n2c(-c3cc4ccccc4s3)cc3c2cc(-c2cc4ccc...,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2633,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.769231,0.0,1.0,0.0,0.0,1.0,0.0,0.0,N#Cc1ccc(-n2c(=O)c3cc4c(=O)n(-c5ccc(C#N)c(-c6c...,0.0
2634,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.750000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,N#CC(C#N)=c1cc2c(=C(C#N)C#N)cc1-2,0.0
2635,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.764706,0.0,0.0,0.0,0.0,0.0,0.0,0.0,N#CC(C#N)=c1ccc(=C(C#N)C#N)c2c1C2,0.0
2636,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.777778,0.0,0.0,0.0,0.0,0.0,0.0,0.0,CC1c2c1c(=C(C#N)C#N)ccc2=C(C#N)C#N,0.0


In [25]:
X_test = test_data_features.drop(columns=['capacity_max', 'smiles'])
results = pipeline.predict_model(model, X_test)

results_smiles = pd.DataFrame({
    'smiles': test_data_features['smiles'],
    'predicted_capacity_max': results
})
results_smiles

,smiles,predicted_capacity_max
0,N#Cc1ccc(C#N)cc1,515.317017
1,N#Cc1ccc(-n2c(-c3c(F)c(F)cc(F)c3F)cc3c2cc(-c2c...,255.881927
2,N#Cc1ccc(-n2c(=O)c3cc4c(=O)n(-c5ccc(C#N)cc5)c(...,245.706055
3,N#Cc1ccc(-n2[nH]c(=O)c3cc4c(=O)[nH]n(-c5ccc(C#...,249.994095
4,N#Cc1ccc(-n2c(-c3cc4ccccc4s3)cc3c2cc(-c2cc4ccc...,565.125854
...,...,...
2633,N#Cc1ccc(-n2c(=O)c3cc4c(=O)n(-c5ccc(C#N)c(-c6c...,236.110291
2634,N#CC(C#N)=c1cc2c(=C(C#N)C#N)cc1-2,365.716339
2635,N#CC(C#N)=c1ccc(=C(C#N)C#N)c2c1C2,377.112915
2636,CC1c2c1c(=C(C#N)C#N)ccc2=C(C#N)C#N,370.189423


In [26]:
results_smiles.sort_values(by='predicted_capacity_max', inplace=True, ascending=False)
results_smiles

,smiles,predicted_capacity_max
4,N#Cc1ccc(-n2c(-c3cc4ccccc4s3)cc3c2cc(-c2cc4ccc...,565.125854
210,N#Cc1ccc(-n2c(-c3cc4ccccc4s3)cc3n(-c4ccc(C#N)c...,555.320435
0,N#Cc1ccc(C#N)cc1,515.317017
11,N#Cc1ccc(C#N)c2nsnc12,507.626770
2626,N#Cc1ccc(C#N)c2c1CC2,500.332611
...,...,...
2483,N#Cc1ccc(-c2ccc(-c3ccc(-c4ccc(-n5c(-c6cc(Br)c(...,119.835236
2486,N#CC=Cc1ccc(-n2c(-c3cccc(C#N)c3)cc3c2cc(-c2c(F...,119.307129
745,Cc1c(-c2ccc(C#N)cc2)sc(-c2ccc(-c3ccc(-c4nc5cc(...,119.019363
2371,N#Cc1ccc(-c2ccc(-c3ccc(-c4cccc(-n5c(-c6ccccc6)...,118.217987


In [27]:
results_smiles.to_csv('results/ceteefy_cof_capacity_predictions.csv', index=False)

In [28]:
ctfs_original = pd.read_csv('../../../data/predictor/ctf/data_substrate_ctfs.csv')
ctfs_original

,smiles,capacity_max
0,N#Cc1ccc(C#N)cc1,628.0
1,CC(C)c1ccc(-n2c(-c3ccc(C#N)cc3)cc3c2cc(-c2ccc(...,255.0
2,Cc1ccc2nc(-n3c(-c4ccc(C#N)cc4)cc4c3cc(-c3ccc(C...,269.0
3,N#Cc1c(F)c(F)c(C#N)c(F)c1F,379.0
4,N#CC(C#N)=c1ccc(=C(C#N)C#N)cc1,383.0
5,N#C/C=C/C#N,404.0
6,N#Cc1ccc(-c2cc3c(cc(-c4ccc(C#N)cc4)n3-c3ccc(C#...,291.0
7,N#Cc1cc(C#N)c2ccc3c(C#N)cc(C#N)c4ccc1c2c43,500.0
8,N#Cc1c2ccccc2c(C#N)c2ccccc12,589.0
9,N#Cc1cncc(C#N)c1,66.0


In [29]:
def canon_smiles(smiles):
    from rdkit import Chem
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)
ctfs_original_smiles = ctfs_original['smiles'].apply(canon_smiles)
ctfs_original_smiles

0                                      N#Cc1ccc(C#N)cc1
1     CC(C)c1ccc(-n2c(-c3ccc(C#N)cc3)cc3c2cc(-c2ccc(...
2     Cc1ccc2nc(-n3c(-c4ccc(C#N)cc4)cc4c3cc(-c3ccc(C...
3                            N#Cc1c(F)c(F)c(C#N)c(F)c1F
4                        N#CC(C#N)=c1ccc(=C(C#N)C#N)cc1
5                                           N#C/C=C/C#N
6     N#Cc1ccc(-c2cc3c(cc(-c4ccc(C#N)cc4)n3-c3ccc(C#...
7            N#Cc1cc(C#N)c2ccc3c(C#N)cc(C#N)c4ccc1c2c43
8                          N#Cc1c2ccccc2c(C#N)c2ccccc12
9                                      N#Cc1cncc(C#N)c1
10                          N#Cc1ccc(-c2ccc(C#N)cc2)cc1
11                          N#Cc1ccc(-c2ccc(C#N)cn2)nc1
12                                     N#Cc1cccc(C#N)n1
13    N#Cc1ccc(-n2[nH]c(=O)c3cc4c(=O)[nH]n(-c5ccc(C#...
14    N#Cc1ccc(-n2c(=O)c3cc4c(=O)n(-c5ccc(C#N)cc5)c(...
15    N#Cc1ccc(-c2cc(-c3ccc(C#N)cc3)nc(-c3ccc(C#N)cc...
16     N#Cc1ccc(-c2nn(-c3ccc(C#N)cc3)c(=O)c3ccccc23)cc1
17           N#Cc1ccc(-c2nc3sc(-c4ccc(C#N)cc4)nc

In [30]:
results_smiles['smiles'] = results_smiles['smiles'].apply(canon_smiles)
results_smiles

,smiles,predicted_capacity_max
4,N#Cc1ccc(-n2c(-c3cc4ccccc4s3)cc3c2cc(-c2cc4ccc...,565.125854
210,N#Cc1ccc(-n2c(-c3cc4ccccc4s3)cc3n(-c4ccc(C#N)c...,555.320435
0,N#Cc1ccc(C#N)cc1,515.317017
11,N#Cc1ccc(C#N)c2nsnc12,507.626770
2626,N#Cc1ccc(C#N)c2c1CC2,500.332611
...,...,...
2483,N#Cc1ccc(-c2ccc(-c3ccc(-c4ccc(-n5c(-c6cc(Br)c(...,119.835236
2486,N#CC=Cc1ccc(-n2c(-c3cccc(C#N)c3)cc3c2cc(-c2c(F...,119.307129
745,Cc1c(-c2ccc(C#N)cc2)sc(-c2ccc(-c3ccc(-c4nc5cc(...,119.019363
2371,N#Cc1ccc(-c2ccc(-c3ccc(-c4cccc(-n5c(-c6ccccc6)...,118.217987


In [32]:
results_smiles_filtered = results_smiles[~results_smiles['smiles'].isin(ctfs_original_smiles)]
results_smiles_filtered

,smiles,predicted_capacity_max
210,N#Cc1ccc(-n2c(-c3cc4ccccc4s3)cc3n(-c4ccc(C#N)c...,555.320435
2626,N#Cc1ccc(C#N)c2c1CC2,500.332611
816,N#CC#Cc1ccc(C#N)cc1,491.868652
84,N#Cc1ccc(C#N)c(Br)c1Br,488.957947
2261,N#Cc1ccc(C#N)c(Cl)c1Cl,486.053406
...,...,...
2483,N#Cc1ccc(-c2ccc(-c3ccc(-c4ccc(-n5c(-c6cc(Br)c(...,119.835236
2486,N#CC=Cc1ccc(-n2c(-c3cccc(C#N)c3)cc3c2cc(-c2c(F...,119.307129
745,Cc1c(-c2ccc(C#N)cc2)sc(-c2ccc(-c3ccc(-c4nc5cc(...,119.019363
2371,N#Cc1ccc(-c2ccc(-c3ccc(-c4cccc(-n5c(-c6ccccc6)...,118.217987


In [33]:
results_smiles_filtered.to_csv('results/ceteefy_cof_capacity_predictions_filtered.csv', index=False)

In [42]:
from rdkit.Chem import PandasTools

results_smiles_filtered = results_smiles_filtered[['smiles', 'predicted_capacity_max']]

PandasTools.AddMoleculeColumnToFrame(results_smiles_filtered, smilesCol='smiles', molCol='molecule')
results_smiles_filtered = results_smiles_filtered[['smiles', 'molecule', 'predicted_capacity_max']]

In [43]:
PandasTools.SaveXlsxFromFrame(results_smiles_filtered, 'ceteefy_capacity_predictions_filtered.xlsx', molCol='molecule')